In [54]:
import pandas as pd
import numpy as np

def read_sheet(sheet, first):
    ''' Read a sheet from the JMP_2021_WLD.xlsx file and return a dataframe
    args.
        sheet: str, name of the sheet to read
        first: bool, whether this is the first sheet to be read
    returns.
        df: pd.DataFrame, the dataframe of the sheet
    '''
    # clean the dataframe of necessary columns
    df = pd.read_excel('initial/JMP_2021_WLD.xlsx', sheet_name=sheet)\
        .drop(columns=['sl','region_who','region_unicef_programme','region_unicef_reporting'], axis=1)\
        .rename({'name': 'country', 'pop_n': 'pop', 'prop_u': 'pop_urban'}, axis=1)\
        .dropna(subset=['country'])\
        .set_index(['country', 'iso3', 'region_sdg', 'year'])
    if not first:
        df = df.drop(columns=['pop', 'pop_urban'], axis=1)
    return df

wat = read_sheet('wat', first=True)
hyg = read_sheet('hyg', first=False)
san = read_sheet('san', first=False)

comb_df = pd.concat([wat, hyg, san], axis=1, join='inner')

cols = [col for col in comb_df.columns if 'arc' not in col]
pop_df = comb_df.copy()
pop_df[cols] = pop_df.loc[:,cols].multiply(comb_df['pop'], axis=0) * 10
pop_df['pop'] = comb_df['pop'] * 1000
pop_df

pop     pop_urban  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  2.077996e+07  4.587799e+06   
                                           2001  2.160699e+07  4.790054e+06   
                                           2002  2.260077e+07  5.031158e+06   
                                           2003  2.368087e+07  5.293385e+06   
                                           2004  2.472669e+07  5.563505e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  1.403034e+07  4.531238e+06   
                                           2017  1.423660e+07  4.589452e+06   
                                           2018  1.443881e+07  4.650597e+06   
                                           2019  1.464547e+07  4.717307e+06   
                                           2020  1.486293e+07  4.792105e+06   

                                                    wat_bas_n     wat_lim_n  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  5.854008e+06  7.606791e+05   
                                           2001  6.093035e+06  7.911491e+05   
                                           2002  6.833657e+06  8.926112e+05   
                                           2003  7.642699e+06  1.003395e+06   
                                           2004  8.488316e+06  1.118861e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  9.045214e+06  1.797998e+06   
                                           2017  9.110943e+06  1.874760e+06   
                                           2018  9.174244e+06  1.951685e+06   
                                           2019  9.240555e+06  2.029876e+06   
                                           2020  9.314070e+06  2.110170e+06   

                                                  wat_unimp_n     wat_sur_n  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  8.972433e+06  5.192837e+06   
                                           2001  9.327207e+06  5.395601e+06   
                                           2002  9.422176e+06  5.452329e+06   
                                           2003  9.522299e+06  5.512479e+06   
                                           2004  9.575232e+06  5.544279e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  2.243020e+06  9.441063e+05   
                                           2017  2.287680e+06  9.632162e+05   
                                           2018  2.331113e+06  9.817697e+05   
                                           2019  2.374697e+06  1.000344e+06   
                                           2020  2.419366e+06  1.019321e+06   

                                                 arc_wat_bas_n     wat_bas_r  \
country     iso3 region_sdg                year                                
Afghanistan AFG  Central and Southern Asia 2000       2.346000  4.444832e+06   
                                           2001       2.346000  4.621735e+06   
                                           2002       2.346000  5.279554e+06   
                                           2003       2.346000  5.998394e+06   
                                           2004       2.346000  6.750433e+06   
...                                                        ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016      -0.460146  7.106665e+06   
                                           2017      -0.460146  7.126328e+06   
                                           2018      -0.460146  7.141851e+06   
                                           2019      -0.460146  7.157469e+06   
              

In [60]:
gdp = pd.read_csv('initial data/API_NY.GDP.MKTP.CD_DS2_en_csv_v2_5358352.csv', header=2)\
    .rename({'Country Code': 'iso3'}, axis=1)\
    .drop(columns=['Country Name', 'Indicator Name', 'Indicator Code', 'Unnamed: 66'], axis=1)
gdp = pd.melt(gdp, id_vars=['iso3'], var_name='year', value_name='gdp')
gdp.year = gdp.year.astype('float64')
all = pop_df.join(gdp.set_index(['iso3', 'year']), on=['iso3', 'year'], how='left')

pop     pop_urban  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  2.077996e+07  4.587799e+06   
                                           2001  2.160699e+07  4.790054e+06   
                                           2002  2.260077e+07  5.031158e+06   
                                           2003  2.368087e+07  5.293385e+06   
                                           2004  2.472669e+07  5.563505e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  1.403034e+07  4.531238e+06   
                                           2017  1.423660e+07  4.589452e+06   
                                           2018  1.443881e+07  4.650597e+06   
                                           2019  1.464547e+07  4.717307e+06   
                                           2020  1.486293e+07  4.792105e+06   

                                                    wat_bas_n     wat_lim_n  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  5.854008e+06  7.606791e+05   
                                           2001  6.093035e+06  7.911491e+05   
                                           2002  6.833657e+06  8.926112e+05   
                                           2003  7.642699e+06  1.003395e+06   
                                           2004  8.488316e+06  1.118861e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  9.045214e+06  1.797998e+06   
                                           2017  9.110943e+06  1.874760e+06   
                                           2018  9.174244e+06  1.951685e+06   
                                           2019  9.240555e+06  2.029876e+06   
                                           2020  9.314070e+06  2.110170e+06   

                                                  wat_unimp_n     wat_sur_n  \
country     iso3 region_sdg                year                               
Afghanistan AFG  Central and Southern Asia 2000  8.972433e+06  5.192837e+06   
                                           2001  9.327207e+06  5.395601e+06   
                                           2002  9.422176e+06  5.452329e+06   
                                           2003  9.522299e+06  5.512479e+06   
                                           2004  9.575232e+06  5.544279e+06   
...                                                       ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016  2.243020e+06  9.441063e+05   
                                           2017  2.287680e+06  9.632162e+05   
                                           2018  2.331113e+06  9.817697e+05   
                                           2019  2.374697e+06  1.000344e+06   
                                           2020  2.419366e+06  1.019321e+06   

                                                 arc_wat_bas_n     wat_bas_r  \
country     iso3 region_sdg                year                                
Afghanistan AFG  Central and Southern Asia 2000       2.346000  4.444832e+06   
                                           2001       2.346000  4.621735e+06   
                                           2002       2.346000  5.279554e+06   
                                           2003       2.346000  5.998394e+06   
                                           2004       2.346000  6.750433e+06   
...                                                        ...           ...   
Zimbabwe    ZWE  Sub-Saharan Africa        2016      -0.460146  7.106665e+06   
                                           2017      -0.460146  7.126328e+06   
                                           2018      -0.460146  7.141851e+06   
                                           2019      -0.460146  7.157469e+06   
              

In [56]:
pop_df.index.dtypes

country       object
iso3          object
region_sdg    object
year           int64
dtype: object

In [57]:
gdp.dtypes

iso3     object
year     object
gdp     float64
dtype: object